In [1]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf



def download_sp500(
    ticker: str = "^GSPC",
    start: str = "2005-01-03",
    end: str = "2022-01-01",
) -> pd.DataFrame:
    """Daily OHLCV with ‘Adj Close’ retained."""
    return yf.download(
        ticker,
        start=start,
        end=end,
        progress=False,
        auto_adjust=False,
    )



def make_clean_table(
    df: pd.DataFrame,
    vol_window: int = 20,
) -> pd.DataFrame:
    """
    Adds derived columns:
        (1)log_ret  : ln(P_t / P_{t-1})
        (2) sigma20  : annualised 20-day realised volatility (√252 · stdev)
    """
    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"

    out = df.copy()
    out.index = out.index.normalize()       # strip intra-day time stamp
    out.index.name = "Date"

    out["log_ret"] = np.log(out[price_col]).diff()
    out["sigma20"] = (
        out["log_ret"]
        .rolling(vol_window)
        .std(ddof=0)
        * np.sqrt(252.0)
    )

    return out.dropna()


def write_csv(
    raw: pd.DataFrame,
    clean: pd.DataFrame,
    stem: str | Path = "sp500_2005_2021",
) -> None:
    """
    Persist two CSV files side-by-side:

        <stem>_raw.csv   ,  <stem>_clean.csv
    """
    stem = Path(stem)
    raw_path   = stem.with_name(f"{stem.stem}_raw.csv")
    clean_path = stem.with_name(f"{stem.stem}_clean.csv")

    raw.to_csv(raw_path, index=True)
    clean.to_csv(clean_path, index=True)

    print("✔  Saved:")
    print(f"   • raw   → {raw_path.resolve()}")
    print(f"   • clean → {clean_path.resolve()}")



if __name__ == "__main__":
    raw_df   = download_sp500()
    clean_df = make_clean_table(raw_df)
    write_csv(raw_df, clean_df)

✔  Saved:
   • raw   → /Users/aqibsyed/Documents/Spring 2025/EE 364b/Final Project/scnn/scnn_zahedi_syed/notebooks/sp500_2005_2021_raw.csv
   • clean → /Users/aqibsyed/Documents/Spring 2025/EE 364b/Final Project/scnn/scnn_zahedi_syed/notebooks/sp500_2005_2021_clean.csv


In [ ]:
"""
sp500_scnn_features.py — Read the clean CSV produced by download_sp500_to_csv.py,
build an AR(p)+volatility feature matrix, and train a Gated-ReLU SCNN to predict
next-day S&P-500 returns.
"""

from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scnn.optimize import optimize
from scnn.regularizers import NeuronGL1



def prepare_from_csv(
    clean_csv: str | Path,
    p: int = 10,
    vol_col: str = "sigma20",
    split_date: str = "2018-01-02",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load the *clean* CSV, build lag features + volatility column, then
    chronologically split into train / test blocks.

    Returns
    -------
    X_train, y_train, X_test, y_test : np.ndarray
        Shapes  (N_train, d) , (N_train, 1) , (N_test, d) , (N_test, 1)
    """
    df = (
        pd.read_csv(clean_csv, index_col=0, parse_dates=True)
        .rename(columns=str.strip)
    )

    log_ret = df["log_ret"]

    # this creates the lag matrix 
    lags = {f"lag_{k}": log_ret.shift(k) for k in range(1, p + 1)}
    lags["vol"] = df[vol_col].shift(1)          # t-1 volatility (information set)

    X = pd.concat(lags, axis=1).dropna()
    y = log_ret.loc[X.index]

    mask = X.index < split_date

    def as_float32(x: pd.DataFrame | pd.Series) -> np.ndarray:
        a = x.to_numpy(dtype=np.float32)
        return a.reshape(-1, 1) if a.ndim == 1 else a

    return (
        as_float32(X[mask]),
        as_float32(y[mask]),
        as_float32(X[~mask]),
        as_float32(y[~mask]),
    )



# nika's code modifed to work with my data
def main() -> None:
    # paths / parameters
    clean_csv   = "sp500_clean_2005_2021.csv"   
    p           = 10
    max_neurons = 300
    lam_gl1     = 1e-3
    huber_delta = 1.0

    # data!!
    X_tr, y_tr, X_te, y_te = prepare_from_csv(clean_csv, p=p)

    # model stuff
    model, _ = optimize(
        formulation="gated_relu",
        max_neurons=max_neurons,
        X_train=X_tr,
        y_train=y_tr,
        X_test=X_te,
        y_test=y_te,
        loss_type="huber",
        huber_delta=huber_delta,
        regularizer=NeuronGL1(lam_gl1),
        verbose=True,
        device="cpu",
    )

    # evaluation
    preds     = model(X_te)
    sign_acc  = (np.sign(preds).ravel() == np.sign(y_te).ravel()).mean()
    print(f"\nTest sign-accuracy : {sign_acc:6.2%}")
    print(f"Train samples      : {len(X_tr):,}")
    print(f"Test  samples      : {len(X_te):,}")
    print(f"Feature dimension  : {X_tr.shape[1]}")


if __name__ == "__main__":
    main()
